# Creador de Mundos RPG — Pack PBR + Skyboxes 360° (Opciones 2 y 3)

Genera en GPU T4 con modelos open-source gratuitos:
- **Opción 2 (Relieve PBR):** `depth-anything/Depth-Anything-V2-Small-hf` calcula mapas **`Normal` + `Roughness`** sin costuras para las **15 capas del terreno** y las **81 texturas** de pueblos, mazmorras y decorado.
- **Opción 3 (Cielos 360° + reflejos PMREM):** `Lykon/dreamshaper-xl-v2-turbo` con padding circular horizontal genera **3 cielos panorámicos equirectangulares 2048×1024** (`dia`, `atardecer`, `noche`).

**Ejecución en 4 celdas (~2-3 minutos total en T4).**

In [ ]:
# 1. Verificar GPU, montar Google Drive y comprobar archivos fuente
import os
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
from google.colab import drive
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/Creador-de-mundos-Rpg-Colab'
SCRIPT_PATH = f'{WORK_DIR}/tools/generar-pbr-y-cielo-ia.py'
SRC_ZIP = f'{WORK_DIR}/assets/texturas_fuente_pbr.zip'
OUT_DIR = '/content/pbr_out'
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(SCRIPT_PATH), f'Falta {SCRIPT_PATH}'
assert os.path.exists(SRC_ZIP), f'Falta {SRC_ZIP}'
print('Drive listo:', WORK_DIR)

In [ ]:
# 2. Instalar dependencias (pin pillow<13 para evitar conflicto en caliente con gradio)
!pip -q install -U diffusers transformers accelerate safetensors "pillow<13"

In [ ]:
# 3. Descomprimir texturas fuente y generar mapas PBR + 3 skyboxes 360°
!rm -rf /content/fuente_tex && mkdir -p /content/fuente_tex
!unzip -q -o "{SRC_ZIP}" -d /content/fuente_tex
!python "{SCRIPT_PATH}" --src /content/fuente_tex --out "{OUT_DIR}" --only pbr,cielos --sky-steps 8 --sky-guidance 2.2

In [ ]:
# 4. Copiar a Drive, mostrar vista previa y descargar ZIP
import base64, json, shutil
from IPython.display import HTML, display
from google.colab import files

DRIVE_OUT = f'{WORK_DIR}/assets/pbr'
shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)

with open(f'{OUT_DIR}/manifest.json', 'r', encoding='utf-8') as f:
    m = json.load(f)
print('Manifest:', {'terreno': bool(m['terreno']), 'texturas': len(m['texturas']), 'cielos': len(m['cielos'])})

muestras = [
    f'{OUT_DIR}/terreno/terreno_normal.webp',
    f'{OUT_DIR}/piedra/t01_n.webp',
    f'{OUT_DIR}/mpared/t01_n.webp',
    f'{OUT_DIR}/cielos/dia.webp',
    f'{OUT_DIR}/cielos/atardecer.webp',
    f'{OUT_DIR}/cielos/noche.webp',
]
for p in muestras:
    if os.path.exists(p):
        print(' ->', os.path.relpath(p, OUT_DIR))
        with open(p, 'rb') as fp:
            b64 = base64.b64encode(fp.read()).decode('ascii')
        display(HTML(f'<img src="data:image/webp;base64,{b64}" width="380" style="border-radius:6px;margin:4px 0">'))

zip_path = shutil.make_archive('/content/pbr_y_cielos_valdris', 'zip', OUT_DIR)
print('ZIP listo:', zip_path)
files.download(zip_path)